# Notebook 03 — prospective external validation of CREB5 in AD astrocytes

This notebook contains the final external-validation path for GSE188545, GSE138852, and GSE174367.

The original working notebook contained failed environment repairs, superseded donor-linkage attempts, and an obsolete period in which GSE138852 was temporarily parked. Those branches are not part of the final analysis. The public version keeps the final donor-level workflow, verifies the frozen provenance artifacts, and reproduces the accession-level statistics and three-study synthesis.

By default the notebook reuses the frozen pseudobulk and genome-wide result files already present in the project folder. Set `REFIT_DE = True` only for a clean DESeq2 rerun from the frozen pseudobulk matrices.

In [ ]:
from pathlib import Path
import hashlib, json, math
import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

PROJECT = Path("/content/drive/MyDrive/AD_Astrocyte_Paper_01")
EV = PROJECT / "external_validation"

PROTOCOL_SHA256 = "ea54a1b3e8e2ff361ec1fc6f55ed2330410e347081926cb657f4df8892e8d8b0"
RECONSTRUCTION_MANIFEST_SHA256 = "bfc48bad2ed73dbcff44dfb8fd50c03c45bf8616600b4fac4fc977e9f9119af7"

REFIT_DE = False
REBUILD_GSE138852_LINKAGE = False

In [ ]:
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass

assert PROJECT.exists(), f"Project folder not found: {PROJECT}"

## Shared helpers

In [ ]:
def sha256_file(path, block_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def load_json(path):
    with open(path) as f:
        return json.load(f)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing required artifact(s):\n" + "\n".join(missing))

In [ ]:
PROTOCOL = EV / "EXTERNAL_VALIDATION_PROTOCOL_LOCK_v1.json"
MANIFEST = EV / "EXTERNAL_VALIDATION_ARTIFACT_RECONSTRUCTION_MANIFEST_v1.json"

require(PROTOCOL)
protocol = load_json(PROTOCOL)

assert protocol["scientific_rules_sha256"] == PROTOCOL_SHA256
assert protocol["processing_order"] == ["GSE188545", "GSE138852", "GSE174367"]

if MANIFEST.exists():
    assert sha256_file(MANIFEST) == RECONSTRUCTION_MANIFEST_SHA256

print("External-validation protocol verified.")
print("Order:", " -> ".join(protocol["processing_order"]))

## DESeq2 helper

The target gene is not queried inside this function. Complete genome-wide results are produced first; CREB5 is extracted later in a separate section.

In [ ]:
def run_deseq2(counts, metadata, design, out_path, feature_info=None):
    from contextlib import redirect_stdout
    import io
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats

    counts = counts.astype(np.int64)
    assert counts.index.equals(metadata.index)
    assert (counts.to_numpy() >= 0).all()

    dds = DeseqDataSet(
        counts=counts,
        metadata=metadata,
        design=design,
        refit_cooks=True,
        n_cpus=2,
        quiet=False,
    )
    dds.deseq2()

    stats = DeseqStats(
        dds,
        contrast=["diagnosis", "AD", "Control"],
        alpha=0.05,
        cooks_filter=True,
        independent_filter=True,
        n_cpus=2,
        quiet=False,
    )

    with redirect_stdout(io.StringIO()):
        stats.summary()

    res = stats.results_df.copy()
    res.index = res.index.astype(str)
    res.index.name = "gene_id"

    if feature_info is not None:
        res = res.join(feature_info, how="left")

    res["CI95_low"] = res["log2FoldChange"] - 1.96 * res["lfcSE"]
    res["CI95_high"] = res["log2FoldChange"] + 1.96 * res["lfcSE"]
    res.reset_index().to_csv(
        out_path,
        index=False,
        compression={"method": "gzip", "mtime": 0},
    )
    return res

## GSE188545

In [ ]:
d = EV / "GSE188545"

PB = d / "GSE188545_astrocyte_pseudobulk_raw_counts_filtered_v1.csv.gz"
META = d / "GSE188545_pseudobulk_donor_metadata_v1.csv"
PB_LOCK = d / "GSE188545_PSEUDOBULK_GENE_DESIGN_LOCK_v1.json"
DE = d / "GSE188545_astrocyte_pseudobulk_DESeq2_genomewide_v1.csv.gz"

require(PB, META, PB_LOCK)
lock_188545 = load_json(PB_LOCK)

assert lock_188545["genes_retained_for_genome_wide_DE"] == 16147
assert lock_188545["design_formula"] == "~ diagnosis"
assert lock_188545["gene_universe_frozen_before_target_reveal"] is True

print("GSE188545:", lock_188545["donors_by_diagnosis"])
print("Accepted astrocytes:", lock_188545["accepted_astrocytes"])

In [ ]:
if REFIT_DE:
    pb = pd.read_csv(PB, compression="gzip")
    meta = pd.read_csv(META, dtype={"donor_id": str})

    donor_order = meta["donor_id"].astype(str).tolist()
    counts = pb.set_index("gene_id")[donor_order].T.astype(np.int64)
    counts.index = pd.Index(donor_order, dtype=object)

    clinical = meta.set_index("donor_id").loc[donor_order, ["diagnosis"]].copy()
    clinical["diagnosis"] = pd.Categorical(
        clinical["diagnosis"], categories=["Control", "AD"], ordered=True
    )

    feature_info = pb.set_index("gene_id")[["gene_symbol", "feature_type"]]
    run_deseq2(counts, clinical, "~diagnosis", DE, feature_info)
else:
    require(DE)

In [ ]:
r188545 = pd.read_csv(DE, compression="gzip")
row = r188545.loc[r188545["gene_symbol"].eq("CREB5")].iloc[0]

gse188545 = {
    "accession": "GSE188545",
    "log2FC": float(row["log2FoldChange"]),
    "SE": float(row["lfcSE"]),
    "p": float(row["pvalue"]),
    "FDR": float(row["padj"]),
}

assert np.isclose(gse188545["log2FC"], 1.231370, atol=5e-6)
assert np.isclose(gse188545["SE"], 0.320057, atol=5e-6)
assert np.isclose(gse188545["p"], 0.000119404244, rtol=1e-6)

gse188545

## GSE138852 — donor reconstruction

In [ ]:
d = EV / "GSE138852"

AUTHOR = d / "GSE138852_author_linked_adsn_metadata_v1.txt"
GEO = d / "GSE138852_covariates.csv.gz"
LINK_LOCK = d / "GSE138852_DONOR_LINKAGE_ELIGIBILITY_LOCK_v1.json"
PB = d / "GSE138852_astrocyte_pseudobulk_raw_counts_filtered_v1.csv.gz"
META = d / "GSE138852_pseudobulk_donor_metadata_v1.csv"
PB_LOCK = d / "GSE138852_PSEUDOBULK_GENE_DESIGN_LOCK_v1.json"
DE = d / "GSE138852_astrocyte_pseudobulk_DESeq2_genomewide_v1.csv.gz"

require(LINK_LOCK, PB, META, PB_LOCK)
link = load_json(LINK_LOCK)

assert link["exact_reconstructed_cell_identifier_match"] is True
assert link["eligible_donors_by_diagnosis"] == {"AD": 6, "Control": 6}
assert link["assigned_astrocytes"] == 2129
assert link["DE_model"] == "~ diagnosis"

print("GSE138852 donor linkage verified.")
print("Assigned nuclei:", link["assigned_nuclei"])
print("Unassigned nuclei:", link["unassigned_nuclei"])

In [ ]:
if REBUILD_GSE138852_LINKAGE:
    require(AUTHOR, GEO)

    author = pd.read_csv(AUTHOR, sep="\t", dtype=str)
    geo = pd.read_csv(GEO, compression="gzip", dtype=str)

    author["barcode_16"] = author["cell"].str.extract(r"^([ACGT]{16})_", expand=False)
    author["GEO_cell_id"] = author["barcode_16"] + "_" + author["batch"]
    geo["GEO_cell_id"] = geo["Unnamed: 0"].astype(str).str.strip()

    assert author["GEO_cell_id"].is_unique
    assert geo["GEO_cell_id"].is_unique
    assert set(author["GEO_cell_id"]) == set(geo["GEO_cell_id"])

    author["donor_assigned"] = author["patient"].str.fullmatch(r"(?:AD|Ct)[1-6]", na=False)
    author["diagnosis"] = np.select(
        [
            author["patient"].str.fullmatch(r"AD[1-6]", na=False),
            author["patient"].str.fullmatch(r"Ct[1-6]", na=False),
        ],
        ["AD", "Control"],
        default="Unassigned",
    )
    author["accepted_astrocyte"] = author["donor_assigned"] & author["cellType"].eq("astro")

    donors = (
        author.loc[author["donor_assigned"]]
        .groupby(["patient", "diagnosis"], observed=True)
        .agg(nuclei=("cell", "size"), astrocytes=("accepted_astrocyte", "sum"))
        .reset_index()
    )

    assert donors["astrocytes"].ge(20).all()
    assert donors["diagnosis"].value_counts().to_dict() == {"AD": 6, "Control": 6}
    display(donors)

In [ ]:
if REFIT_DE:
    from anndata import AnnData
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    from contextlib import redirect_stdout
    import io

    pb = pd.read_csv(PB, compression="gzip", low_memory=False)
    meta = pd.read_csv(META, dtype={"donor_id": str})

    donor_order = meta["donor_id"].astype(str).tolist()
    genes = [c for c in pb.columns if c != "donor_id"]
    counts = pb.set_index("donor_id").loc[donor_order, genes].astype(np.int64)
    counts.index = pd.Index(donor_order, dtype=object)
    counts.columns = pd.Index(genes, dtype=object)

    clinical = meta.set_index("donor_id").loc[donor_order, ["diagnosis"]].copy()
    clinical["diagnosis"] = pd.Categorical(
        clinical["diagnosis"], categories=["Control", "AD"], ordered=True
    )

    adata = AnnData(
        X=counts.to_numpy(),
        obs=clinical,
        var=pd.DataFrame({"feature_id": genes}, index=pd.Index(genes, dtype=object)),
    )
    dds = DeseqDataSet(adata=adata, design="~diagnosis", refit_cooks=True, n_cpus=2)
    dds.deseq2()

    stats = DeseqStats(
        dds,
        contrast=["diagnosis", "AD", "Control"],
        alpha=0.05,
        cooks_filter=True,
        independent_filter=True,
        n_cpus=2,
    )
    with redirect_stdout(io.StringIO()):
        stats.summary()

    res = stats.results_df.copy()
    res.index = res.index.astype(str)
    res.index.name = "feature_id"
    res["CI95_low"] = res["log2FoldChange"] - 1.96 * res["lfcSE"]
    res["CI95_high"] = res["log2FoldChange"] + 1.96 * res["lfcSE"]
    res.reset_index().to_csv(
        DE, index=False, compression={"method": "gzip", "mtime": 0}
    )
else:
    require(DE)

In [ ]:
r138852 = pd.read_csv(DE, compression="gzip")
id_col = "feature_id" if "feature_id" in r138852.columns else "gene_id"
row = r138852.loc[r138852[id_col].astype(str).eq("CREB5")].iloc[0]

gse138852 = {
    "accession": "GSE138852",
    "log2FC": float(row["log2FoldChange"]),
    "SE": float(row["lfcSE"]),
    "p": float(row["pvalue"]),
    "FDR": float(row["padj"]),
}

assert np.isclose(gse138852["log2FC"], 0.426928, atol=5e-6)
assert np.isclose(gse138852["SE"], 0.439374, atol=5e-6)
assert np.isclose(gse138852["p"], 0.331212931908, rtol=1e-6)

gse138852

## GSE174367

In [ ]:
d = EV / "GSE174367"

SOURCE_LOCK = d / "GSE174367_SOURCE_RECONCILIATION_AMENDMENT_v1.json"
PB_LOCK = d / "GSE174367_PSEUDOBULK_GENE_DESIGN_LOCK_v1.json"
PB = d / "GSE174367_astrocyte_pseudobulk_raw_counts_filtered_v1.csv.gz"
META = d / "GSE174367_pseudobulk_donor_metadata_v1.csv"

PRIMARY = d / "GSE174367_DESeq2_primary_genomewide_v1.csv.gz"
PMI = d / "GSE174367_DESeq2_plus_PMI_genomewide_v1.csv.gz"
BATCH = d / "GSE174367_DESeq2_plus_batch_genomewide_v1.csv.gz"

require(SOURCE_LOCK, PB_LOCK, PB, META)
source_lock = load_json(SOURCE_LOCK)
pb_lock = load_json(PB_LOCK)

assert source_lock["H5_barcodes"] == 61770
assert source_lock["final_metadata_cells"] == 61472
assert source_lock["intersection"] == 61472
assert source_lock["H5_only"] == 298

assert pb_lock["accepted_author_ASC_nuclei"] == 4756
assert pb_lock["donors"] == 18
assert pb_lock["genes_retained_for_genome_wide_DE"] == 17289

print("GSE174367 source reconciliation and frozen pseudobulk verified.")

In [ ]:
def prepare_gse174367():
    pb = pd.read_csv(PB, compression="gzip")
    donors = pd.read_csv(
        META,
        dtype={"SampleID": str, "Diagnosis": str, "Sex": str, "Batch": str},
    )

    for c in ["Age", "RIN", "PMI"]:
        donors[c] = pd.to_numeric(donors[c], errors="coerce" if c == "PMI" else "raise")

    order = donors["SampleID"].astype(str).tolist()
    genes = pb["gene_id"].astype(str).tolist()
    counts = pb.set_index("gene_id")[order].T.astype(np.int64)

    info = pb.set_index("gene_id")[["gene_symbol", "feature_type"]]
    return counts, donors.set_index("SampleID").loc[order], info

counts_174367, donors_174367, info_174367 = prepare_gse174367()
assert counts_174367.shape == (18, 17289)

In [ ]:
def metadata_174367(d, include_pmi=False, include_batch=False):
    m = pd.DataFrame(index=d.index.copy())

    m["age_c"] = d["Age"].astype(float) - d["Age"].astype(float).mean()
    m["sex"] = pd.Categorical(d["Sex"], categories=["F", "M"], ordered=True)
    m["rin_c"] = d["RIN"].astype(float) - d["RIN"].astype(float).mean()

    if include_pmi:
        m["pmi_c"] = d["PMI"].astype(float) - d["PMI"].astype(float).mean()

    if include_batch:
        m["batch"] = pd.Categorical(d["Batch"], categories=["1", "2", "3"], ordered=True)

    m["diagnosis"] = pd.Categorical(
        d["Diagnosis"], categories=["Control", "AD"], ordered=True
    )
    return m

In [ ]:
if REFIT_DE:
    m = metadata_174367(donors_174367)
    run_deseq2(
        counts_174367, m,
        "~age_c + sex + rin_c + diagnosis",
        PRIMARY, info_174367
    )

    keep = donors_174367["PMI"].notna()
    d_pmi = donors_174367.loc[keep]
    m_pmi = metadata_174367(d_pmi, include_pmi=True)
    run_deseq2(
        counts_174367.loc[d_pmi.index], m_pmi,
        "~age_c + sex + rin_c + pmi_c + diagnosis",
        PMI, info_174367
    )

    m_batch = metadata_174367(donors_174367, include_batch=True)
    run_deseq2(
        counts_174367, m_batch,
        "~age_c + sex + rin_c + batch + diagnosis",
        BATCH, info_174367
    )
else:
    require(PRIMARY, PMI, BATCH)

In [ ]:
def creb5_from_table(path):
    df = pd.read_csv(path, compression="gzip")
    if "gene_symbol" in df.columns:
        hit = df.loc[df["gene_symbol"].astype(str).eq("CREB5")]
    else:
        key = "feature_id" if "feature_id" in df.columns else "gene_id"
        hit = df.loc[df[key].astype(str).eq("CREB5")]
    assert len(hit) == 1
    r = hit.iloc[0]
    return {
        "log2FC": float(r["log2FoldChange"]),
        "SE": float(r["lfcSE"]),
        "p": float(r["pvalue"]),
        "FDR": float(r["padj"]),
    }

gse174367 = {"accession": "GSE174367", **creb5_from_table(PRIMARY)}
gse174367_pmi = creb5_from_table(PMI)
gse174367_batch = creb5_from_table(BATCH)

assert np.isclose(gse174367["log2FC"], 0.309604, atol=5e-6)
assert np.isclose(gse174367["SE"], 0.299697, atol=5e-6)
assert np.isclose(gse174367["p"], 0.301577233719, rtol=1e-6)

pd.DataFrame([
    {"model": "Primary", **gse174367},
    {"model": "+ PMI", **gse174367_pmi},
    {"model": "+ Batch", **gse174367_batch},
])

## Prespecified three-accession synthesis

In [ ]:
external = pd.DataFrame([gse188545, gse138852, gse174367])
external

In [ ]:
def holm_adjust(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    m = len(p)

    adjusted_ranked = np.empty(m)
    running = 0.0
    for i, value in enumerate(ranked):
        candidate = (m - i) * value
        running = max(running, candidate)
        adjusted_ranked[i] = min(running, 1.0)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_ranked
    return adjusted

external["Holm_p"] = holm_adjust(external["p"])
external

In [ ]:
y = external["log2FC"].to_numpy(float)
se = external["SE"].to_numpy(float)
w = 1 / se**2

fixed = np.sum(w * y) / np.sum(w)
fixed_se = np.sqrt(1 / np.sum(w))
fixed_z = fixed / fixed_se
fixed_p = 2 * norm.sf(abs(fixed_z))
fixed_ci = (fixed - 1.96 * fixed_se, fixed + 1.96 * fixed_se)

Q = np.sum(w * (y - fixed)**2)
df = len(y) - 1
Q_p = chi2.sf(Q, df)
I2 = max(0.0, (Q - df) / Q) * 100 if Q > 0 else 0.0

C = np.sum(w) - np.sum(w**2) / np.sum(w)
tau2 = max(0.0, (Q - df) / C)
wr = 1 / (se**2 + tau2)

random = np.sum(wr * y) / np.sum(wr)
random_se = np.sqrt(1 / np.sum(wr))
random_z = random / random_se
random_p = 2 * norm.sf(abs(random_z))
random_ci = (random - 1.96 * random_se, random + 1.96 * random_se)

In [ ]:
assert np.isclose(fixed, 0.677998, atol=5e-6)
assert np.isclose(fixed_se, 0.195831, atol=5e-6)
assert np.isclose(fixed_p, 0.000535861203005, rtol=1e-6)
assert np.isclose(Q, 4.826886, atol=5e-6)
assert np.isclose(I2, 58.57, atol=0.02)
assert np.isclose(tau2, 0.169902, atol=5e-6)
assert np.isclose(random, 0.670385, atol=5e-6)
assert np.isclose(random_se, 0.311931, atol=5e-6)
assert np.isclose(random_p, 0.0316230963615, rtol=1e-6)

summary = pd.DataFrame([
    {
        "analysis": "Fixed effect",
        "log2FC": fixed,
        "SE": fixed_se,
        "CI_low": fixed_ci[0],
        "CI_high": fixed_ci[1],
        "P": fixed_p,
    },
    {
        "analysis": "DerSimonian-Laird random sensitivity",
        "log2FC": random,
        "SE": random_se,
        "CI_low": random_ci[0],
        "CI_high": random_ci[1],
        "P": random_p,
    },
])

print(external[["accession", "log2FC", "SE", "p", "Holm_p"]].to_string(index=False))
print()
print(summary.to_string(index=False))
print(f"Q={Q:.6f}, df={df}, Q P={Q_p:.6g}, I²={I2:.2f}%, tau²={tau2:.6f}")

## Final checks

All three primary external-validation estimates are positive. GSE188545 remains significant after Holm correction. The fixed-effect pooled estimate is positive and statistically significant; the DerSimonian–Laird sensitivity analysis is also positive. No regional estimates from SEA-AD or GSE160936 are pooled here because those analyses contain within-cohort or paired-donor dependencies.

In [ ]:
expected_holm = {
    "GSE188545": 0.000358212733443,
    "GSE138852": 0.603154467438,
    "GSE174367": 0.603154467438,
}

for accession, expected in expected_holm.items():
    observed = external.loc[external["accession"].eq(accession), "Holm_p"].iloc[0]
    assert np.isclose(observed, expected, rtol=1e-9, atol=1e-12)

assert (external["log2FC"] > 0).all()
print("Notebook 03 final validation checks: PASS")